In [ ]:
# Fase 3 baseline: duck harness (Tufa Labs) en la G4 — validación offline CORTA.
import json, os, pickle, subprocess, sys, sysconfig, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
# CUDA linker path para vLLM/torch en imagen Kaggle
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
# Corte de validación offline (minutos) para NO gastar 9h de G4 cuando no es rerun.
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "70"))
WORKING = Path("/kaggle/working"); WORKING.mkdir(parents=True, exist_ok=True)
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


In [ ]:
# Instalar arc-agi del wheelhouse de la competencia (offline)
COMP_ROOT = None
for dp, dn, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dn:
        COMP_ROOT = Path(dp); break
assert COMP_ROOT, "wheelhouse no encontrado"
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi; print("arc_agi OK")

# Localizar el bundle del solver por su marker
BUNDLE = None
for m in Path("/kaggle/input").rglob("taaf-kaggle-bundle.json"):
    BUNDLE = m.parent; break
assert BUNDLE, "bundle TAAF no encontrado (adjunta thtennant/taaf-kaggle-source-share-fork)"
print("BUNDLE =", BUNDLE)

# Mapear datasets adjuntos a sus mounts
DATASET_SOURCES = ["thtennant/taaf-kaggle-source-share-fork",
                   "driessmit1/arc3-vllm-h100-wheelhouse-v3",
                   "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
def mount(ref):
    o,s = ref.split("/",1)
    for c in (Path("/kaggle/input")/s, Path("/kaggle/input/datasets")/o/s):
        if c.exists(): return str(c)
    return str(Path("/kaggle/input")/s)
paths = {r: (str(BUNDLE) if i==0 else mount(r)) for i,r in enumerate(DATASET_SOURCES)}
env_extra = {"TAAF_KAGGLE_INPUT_PATHS": json.dumps(paths, sort_keys=True),
             "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
             "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([])}
os.environ.update(env_extra)
SETUP_ENV = WORKING/"taaf_setup_env.json"; SETUP_ENV.write_text(json.dumps(env_extra))
print(paths)


In [ ]:
# Importar repos del bundle y correr setup_commands (instala vLLM, arranca el server)
def source_entries(b):
    out=[]
    for repo in sorted((b/"src").iterdir(), reverse=True):
        for c in (repo/"src", repo):
            if c.is_dir(): out.append(c)
    return out
entries = source_entries(BUNDLE)
for e in entries: sys.path.insert(0, str(e))
pth = Path(sysconfig.get_paths()["purelib"])/"taaf_sources.pth"
pth.write_text("".join(f"{e}\n" for e in entries))

def cmd_env():
    env = os.environ.copy(); env["PYTHON"]=sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"]=str(BUNDLE); env["TAAF_KAGGLE_WORKING_DIR"]=str(WORKING)
    env["TAAF_KAGGLE_SETUP_ENV"]=str(SETUP_ENV)
    env.update({str(k):str(v) for k,v in json.loads(SETUP_ENV.read_text()).items()})
    return env
env = cmd_env()
for c in json.loads((BUNDLE/"setup_commands.json").read_text()):
    print("setup:", c[:80], flush=True)
    subprocess.run(c, shell=True, check=True, cwd=WORKING, env=env)
    env = cmd_env(); os.environ.update(env)
for e in reversed([x for x in os.environ.get("PYTHONPATH","").split(os.pathsep) if x]):
    if e not in sys.path: sys.path.insert(0, e)
print("setup completo")


In [ ]:
# Cargar benchmark + target, jugar (offline recortado / gateway en rerun)
with open(BUNDLE/"deploy_target.pkl","rb") as f: target = pickle.load(f)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION
with open(BUNDLE/"benchmark_initial.pkl","rb") as f: bm = pickle.load(f)
bm.job_dir = WORKING; bm.n_passes = 1; bm.game_weights = None
os.environ.setdefault("RECORDINGS_DIR", str(WORKING/"server_recording"))

# Graft install (base = config v12 de thtennant, que marco 1.17) + schema_helpers:
# precarga helpers de analisis testeados (grid_diff, connected_components,
# action_effect_summary, recent_history) en el sandbox python del agente — el 27B
# reescribe esa plomeria con bugs en cada juego. NUESTRA tesis de feature injection,
# implementada por el autor del fork como graft sin habilitar (WP3).
# goalkeep NO va: marco 0.81 en el set oculto (-0.36 vs v12; ver working notes
# 2026-08-12). Blindado: cualquier fallo -> stock.
# Verificado en CPU local con scripts/smoke_graft_install.py (banner + prelude 8KB).
try:
    from taaf_grafts.composite import install as _graft_install
    _graft_install(bm, flags={"efficiency": True, "retry_guard": True,
                              "shortcircuit": True, "schema_helpers": True})
except Exception as exc:
    print(f"[taaf_grafts] graft failed, running stock: {type(exc).__name__}: {exc}")

# AMPLIFICACION (nuestro diferencial): anadimos los helpers de navegacion al mismo
# prelude que schema_helpers inyecta en el sandbox del agente. schema_helpers lee
# SANDBOX_HELPERS_PRELUDE y HELPERS_PROMPT_NOTE en CADA llamada (no los captura al
# importar), asi que extenderlos aqui basta. Verificado en CPU con
# scripts/test_sandbox_nav.py (compila bajo SAFE_BUILTINS, aprende el modelo de
# movimiento, descarta acciones erraticas, degrada a vacio sin transiciones).
try:
    import base64 as _b64
    import taaf_grafts.schema_helpers as _sh
    _nav_src = _b64.b64decode("ZGVmIF9uYXZfYXNfZ3JpZCh4KToKICAgICIiIkNvbnZpZXJ0ZSBmcmFtZSBvIGxpc3RhIGRlIGxpc3RhcyBlbiBncmlkOyBOb25lIHNpIG5vIHNlIHB1ZWRlLiIiIgogICAgaWYgeCBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICBncmlkID0gTm9uZQogICAgZm9yIG5hbWUgaW4gKCJfZ3JpZCIsICJncmlkIik6CiAgICAgICAgdmFsdWUgPSBnZXRhdHRyKHgsIG5hbWUsIE5vbmUpCiAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgICAgIGdyaWQgPSB2YWx1ZQogICAgICAgICAgICBicmVhawogICAgaWYgZ3JpZCBpcyBOb25lIGFuZCBpc2luc3RhbmNlKHgsIChsaXN0LCB0dXBsZSkpOgogICAgICAgIGdyaWQgPSB4CiAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJvd3MgPSBbXQogICAgZm9yIHJvdyBpbiBncmlkOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdywgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcm93cy5hcHBlbmQobGlzdChyb3cpKQogICAgcmV0dXJuIHJvd3MKCmRlZiBfbmF2X3NoaWZ0KGJlZm9yZSwgYWZ0ZXIpOgogICAgIiIiRGVzcGxhemFtaWVudG8gKGRyb3csIGRjb2wpIGRlbCBvYmpldG8gcXVlIHNlIG1vdmlvIGVudHJlIGRvcyBncmlkcy4KCiAgICBNZXRvZG86IHBvciBjYWRhIGNvbG9yLCBjZW50cm9pZGUgZGUgbGFzIGNlbGRhcyBxdWUgbG8gR0FOQVJPTiBtZW5vcyBjZW50cm9pZGUKICAgIGRlIGxhcyBxdWUgbG8gUEVSRElFUk9OLiBTZSB0b21hIGVsIGNvbG9yIGNvbiBtYXMgY2VsZGFzIGVtcGFyZWphZGFzLiBEZXZ1ZWx2ZQogICAgTm9uZSBzaSBuYWRhIHNlIG1vdmlvIG8gc2kgZWwgY2FtYmlvIG5vIGVzIHVuYSB0cmFzbGFjaW9uIGxpbXBpYS4KICAgICIiIgogICAgZ2EgPSBfbmF2X2FzX2dyaWQoYmVmb3JlKQogICAgZ2IgPSBfbmF2X2FzX2dyaWQoYWZ0ZXIpCiAgICBpZiBnYSBpcyBOb25lIG9yIGdiIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgICMgRWwgRk9ORE8gc2UgbXVldmUgYWwgcmV2ZXMgcXVlIGVsIG9iamV0byAoZG9uZGUgZWwgb2JqZXRvIGxsZWdhLCBlbCBmb25kbyBzZQogICAgIyBwaWVyZGUpLiBTaW4gZXhjbHVpcmxvLCBlbCBzaWdubyBkZWwgZGVzcGxhemFtaWVudG8gc2FsZSBpbnZlcnRpZG8gY3VhbmRvIGVsCiAgICAjIGZvbmRvIGdhbmEgZWwgZGVzZW1wYXRlIC0tIGJ1ZyByZWFsIGNhemFkbyBwb3Igc2NyaXB0cy90ZXN0X3NhbmRib3hfbmF2LnB5LgogICAgZnJlcSA9IHt9CiAgICBmb3Igcm93IGluIGdhOgogICAgICAgIGZvciB2IGluIHJvdzoKICAgICAgICAgICAgaWYgdiBpbiBmcmVxOgogICAgICAgICAgICAgICAgZnJlcVt2XSA9IGZyZXFbdl0gKyAxCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBmcmVxW3ZdID0gMQogICAgYmFja2dyb3VuZCA9IE5vbmUKICAgIGJhY2tncm91bmRfbiA9IDAKICAgIGZvciB2IGluIGZyZXE6CiAgICAgICAgaWYgZnJlcVt2XSA+IGJhY2tncm91bmRfbjoKICAgICAgICAgICAgYmFja2dyb3VuZF9uID0gZnJlcVt2XQogICAgICAgICAgICBiYWNrZ3JvdW5kID0gdgogICAgZ2FpbmVkID0ge30KICAgIGxvc3QgPSB7fQogICAgcm93cyA9IG1pbihsZW4oZ2EpLCBsZW4oZ2IpKQogICAgZm9yIHIgaW4gcmFuZ2Uocm93cyk6CiAgICAgICAgcmEgPSBnYVtyXQogICAgICAgIHJiID0gZ2Jbcl0KICAgICAgICBjb2xzID0gbWluKGxlbihyYSksIGxlbihyYikpCiAgICAgICAgZm9yIGMgaW4gcmFuZ2UoY29scyk6CiAgICAgICAgICAgIHZhID0gcmFbY10KICAgICAgICAgICAgdmIgPSByYltjXQogICAgICAgICAgICBpZiB2YSA9PSB2YjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIHZiIGluIGdhaW5lZDoKICAgICAgICAgICAgICAgIGdhaW5lZFt2Yl0uYXBwZW5kKChyLCBjKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGdhaW5lZFt2Yl0gPSBbKHIsIGMpXQogICAgICAgICAgICBpZiB2YSBpbiBsb3N0OgogICAgICAgICAgICAgICAgbG9zdFt2YV0uYXBwZW5kKChyLCBjKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGxvc3RbdmFdID0gWyhyLCBjKV0KICAgIGJlc3QgPSBOb25lCiAgICBiZXN0X24gPSAwCiAgICBmb3IgY29sb3IgaW4gZ2FpbmVkOgogICAgICAgIGlmIGNvbG9yID09IGJhY2tncm91bmQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgY29sb3Igbm90IGluIGxvc3Q6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbiA9IG1pbihsZW4oZ2FpbmVkW2NvbG9yXSksIGxlbihsb3N0W2NvbG9yXSkpCiAgICAgICAgaWYgbiA+IGJlc3RfbjoKICAgICAgICAgICAgYmVzdF9uID0gbgogICAgICAgICAgICBiZXN0ID0gY29sb3IKICAgIGlmIGJlc3QgaXMgTm9uZSBvciBiZXN0X24gPT0gMDoKICAgICAgICByZXR1cm4gTm9uZQogICAgZyA9IGdhaW5lZFtiZXN0XQogICAgbCA9IGxvc3RbYmVzdF0KICAgIGdyID0gc3VtKHBbMF0gZm9yIHAgaW4gZykgLyBsZW4oZykKICAgIGdjID0gc3VtKHBbMV0gZm9yIHAgaW4gZykgLyBsZW4oZykKICAgIGxyID0gc3VtKHBbMF0gZm9yIHAgaW4gbCkgLyBsZW4obCkKICAgIGxjID0gc3VtKHBbMV0gZm9yIHAgaW4gbCkgLyBsZW4obCkKICAgIGRyID0gaW50KHJvdW5kKGdyIC0gbHIpKQogICAgZGMgPSBpbnQocm91bmQoZ2MgLSBsYykpCiAgICBpZiBkciA9PSAwIGFuZCBkYyA9PSAwOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gKGRyLCBkYykKCmRlZiBfbmF2X3RyYW5zaXRpb25zKCk6CiAgICAiIiJBbGNhbnphIGVsIGdsb2JhbCBgdHJhbnNpdGlvbnNgIGRlbCBzYW5kYm94IChzdSBub21icmUgcXVlZGEgc29tYnJlYWRvCiAgICBkZW50cm8gZGUgbGFzIGZ1bmNpb25lcyBxdWUgbG8gcmVjaWJlbiBwb3IgcGFyYW1ldHJvKS4iIiIKICAgIHJldHVybiB0cmFuc2l0aW9ucyAgIyBub3FhOiBGODIxIC0tIGxvIGRlZmluZSBlbCBib290c3RyYXAgZGVsIHNhbmRib3gKCmRlZiBfbmF2X2FjdGlvbl9uYW1lKHQpOgogICAgYSA9IGdldGF0dHIodCwgImFjdGlvbiIsIE5vbmUpCiAgICBpZiBhIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIG5hbWUgPSBnZXRhdHRyKGEsICJuYW1lIiwgTm9uZSkKICAgIGlmIGlzaW5zdGFuY2UobmFtZSwgc3RyKSBhbmQgbmFtZToKICAgICAgICByZXR1cm4gbmFtZQogICAgcmV0dXJuIHN0cihhKQoKZGVmIG1vdGlvbl9tb2RlbCgpOgogICAgIiIiTW9kZWxvIGRlIG1vdmltaWVudG8gTUVESURPIGRlIHR1cyBwcm9waWFzIHRyYW5zaWNpb25lcy4KCiAgICBEZXZ1ZWx2ZSB7bm9tYnJlX2RlX2FjY2lvbjogW2Ryb3csIGRjb2xdfSBzb2xvIHBhcmEgbGFzIGFjY2lvbmVzIHF1ZSBwcm9kdWplcm9uCiAgICB1biBkZXNwbGF6YW1pZW50byBDT05TSVNURU5URSAobGEgbWF5b3JpYSBkZSBzdXMgb2JzZXJ2YWNpb25lcyBjb2luY2lkZW4pLiBMYXMKICAgIGFjY2lvbmVzIHF1ZSBubyBtdWV2ZW4gbmFkYSwgbyBxdWUgbXVldmVuIGRlIGZvcm1hIGVycmF0aWNhLCBxdWVkYW4gZnVlcmEuCiAgICAiIiIKICAgIGNvdW50cyA9IHt9CiAgICB0cnk6CiAgICAgICAgaXRlbXMgPSBfbmF2X3RyYW5zaXRpb25zKCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIHt9CiAgICBmb3IgdCBpbiBpdGVtczoKICAgICAgICBuYW1lID0gX25hdl9hY3Rpb25fbmFtZSh0KQogICAgICAgIGlmIG5vdCBuYW1lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGJlZm9yZSA9IGdldGF0dHIodCwgImJlZm9yZV9mcmFtZSIsIE5vbmUpCiAgICAgICAgYWZ0ZXIgPSBnZXRhdHRyKHQsICJhZnRlcl9mcmFtZSIsIE5vbmUpCiAgICAgICAgc2hpZnQgPSBfbmF2X3NoaWZ0KGJlZm9yZSwgYWZ0ZXIpCiAgICAgICAgaWYgc2hpZnQgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBrZXkgPSBzdHIoc2hpZnRbMF0pICsgIiwiICsgc3RyKHNoaWZ0WzFdKQogICAgICAgIGlmIG5hbWUgbm90IGluIGNvdW50czoKICAgICAgICAgICAgY291bnRzW25hbWVdID0ge30KICAgICAgICBidWNrZXQgPSBjb3VudHNbbmFtZV0KICAgICAgICBpZiBrZXkgaW4gYnVja2V0OgogICAgICAgICAgICBidWNrZXRba2V5XSA9IGJ1Y2tldFtrZXldICsgMQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGJ1Y2tldFtrZXldID0gMQogICAgbW9kZWwgPSB7fQogICAgZm9yIG5hbWUgaW4gY291bnRzOgogICAgICAgIGJ1Y2tldCA9IGNvdW50c1tuYW1lXQogICAgICAgIHRvdGFsID0gc3VtKGJ1Y2tldFtrXSBmb3IgayBpbiBidWNrZXQpCiAgICAgICAgYmVzdF9rZXkgPSBOb25lCiAgICAgICAgYmVzdF9uID0gMAogICAgICAgIGZvciBrIGluIGJ1Y2tldDoKICAgICAgICAgICAgaWYgYnVja2V0W2tdID4gYmVzdF9uOgogICAgICAgICAgICAgICAgYmVzdF9uID0gYnVja2V0W2tdCiAgICAgICAgICAgICAgICBiZXN0X2tleSA9IGsKICAgICAgICBpZiBiZXN0X2tleSBpcyBOb25lIG9yIHRvdGFsID09IDA6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgYmVzdF9uICogMiA8IHRvdGFsOiAgIyBzaW4gbWF5b3JpYSA9PiBtb3ZpbWllbnRvIGVycmF0aWNvLCBzZSBkZXNjYXJ0YQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHBhcnRzID0gYmVzdF9rZXkuc3BsaXQoIiwiKQogICAgICAgIG1vZGVsW25hbWVdID0gW2ludChwYXJ0c1swXSksIGludChwYXJ0c1sxXSldCiAgICByZXR1cm4gbW9kZWwKCmRlZiBwbGFuX21vdmVzKGRyb3csIGRjb2wsIGxpbWl0PTQwKToKICAgICIiIlNlY3VlbmNpYSBkZSBhY2Npb25lcyBxdWUgbG9ncmEgZWwgZGVzcGxhemFtaWVudG8gKGRyb3csIGRjb2wpIHBlZGlkby4KCiAgICBVc2EgZWwgbW9kZWxvIGRlIG1vdmltaWVudG8gbWVkaWRvIHkgYXZhbnphIGNvbiBlbCBwYXNvIHF1ZSBtYXMgcmVkdWNlIGxhCiAgICBkaXN0YW5jaWEgcmVzdGFudGUuIERldnVlbHZlIFtdIHNpIG5vIGhheSBtb2RlbG8gdXRpbCBvIHNpIG5vIHNlIHB1ZWRlIGFjZXJjYXIuCiAgICBQYXNhbGEgZGlyZWN0byBhIGFjdGlvbiguLi4pIHBhcmEgZWplY3V0YXIgTVVDSE9TIHBhc29zIGVuIFVOIHR1cm5vLgogICAgIiIiCiAgICBtb2RlbCA9IG1vdGlvbl9tb2RlbCgpCiAgICBpZiBub3QgbW9kZWw6CiAgICAgICAgcmV0dXJuIFtdCiAgICBtb3ZlcyA9IFtdCiAgICByciA9IGludChkcm93KQogICAgY2MgPSBpbnQoZGNvbCkKICAgIHN0ZXBzID0gMAogICAgd2hpbGUgc3RlcHMgPCBpbnQobGltaXQpIGFuZCAocnIgIT0gMCBvciBjYyAhPSAwKToKICAgICAgICBiZXN0X25hbWUgPSBOb25lCiAgICAgICAgYmVzdF9nYWluID0gMAogICAgICAgIGJlc3RfcnIgPSBycgogICAgICAgIGJlc3RfY2MgPSBjYwogICAgICAgIGZvciBuYW1lIGluIG1vZGVsOgogICAgICAgICAgICB2ID0gbW9kZWxbbmFtZV0KICAgICAgICAgICAgbnIgPSByciAtIHZbMF0KICAgICAgICAgICAgbmMgPSBjYyAtIHZbMV0KICAgICAgICAgICAgZ2FpbiA9IChhYnMocnIpICsgYWJzKGNjKSkgLSAoYWJzKG5yKSArIGFicyhuYykpCiAgICAgICAgICAgIGlmIGdhaW4gPiBiZXN0X2dhaW46CiAgICAgICAgICAgICAgICBiZXN0X2dhaW4gPSBnYWluCiAgICAgICAgICAgICAgICBiZXN0X25hbWUgPSBuYW1lCiAgICAgICAgICAgICAgICBiZXN0X3JyID0gbnIKICAgICAgICAgICAgICAgIGJlc3RfY2MgPSBuYwogICAgICAgIGlmIGJlc3RfbmFtZSBpcyBOb25lOgogICAgICAgICAgICBicmVhawogICAgICAgIG1vdmVzLmFwcGVuZChiZXN0X25hbWUpCiAgICAgICAgcnIgPSBiZXN0X3JyCiAgICAgICAgY2MgPSBiZXN0X2NjCiAgICAgICAgc3RlcHMgPSBzdGVwcyArIDEKICAgIHJldHVybiBtb3ZlcwoKZGVmIHBsYXllcl9wb3MoKToKICAgICIiIlBvc2ljaW9uIFtyb3csIGNvbF0gZGVsIG9iamV0byBxdWUgc2UgbW92aW8gZW4gbGEgdWx0aW1hIHRyYW5zaWNpb24gdXRpbCwKICAgIG8gTm9uZSBzaSBhdW4gbm8gc2UgaGEgb2JzZXJ2YWRvIG5pbmd1biBtb3ZpbWllbnRvLiIiIgogICAgdHJ5OgogICAgICAgIGl0ZW1zID0gX25hdl90cmFuc2l0aW9ucygpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZHggPSBsZW4oaXRlbXMpIC0gMQogICAgd2hpbGUgaWR4ID49IDA6CiAgICAgICAgdCA9IGl0ZW1zW2lkeF0KICAgICAgICBiZWZvcmUgPSBnZXRhdHRyKHQsICJiZWZvcmVfZnJhbWUiLCBOb25lKQogICAgICAgIGFmdGVyID0gZ2V0YXR0cih0LCAiYWZ0ZXJfZnJhbWUiLCBOb25lKQogICAgICAgIGdhID0gX25hdl9hc19ncmlkKGJlZm9yZSkKICAgICAgICBnYiA9IF9uYXZfYXNfZ3JpZChhZnRlcikKICAgICAgICBzaGlmdCA9IF9uYXZfc2hpZnQoYmVmb3JlLCBhZnRlcikKICAgICAgICBpZiBzaGlmdCBpcyBub3QgTm9uZSBhbmQgZ2EgaXMgbm90IE5vbmUgYW5kIGdiIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjZWxscyA9IFtdCiAgICAgICAgICAgIHJvd3MgPSBtaW4obGVuKGdhKSwgbGVuKGdiKSkKICAgICAgICAgICAgZm9yIHIgaW4gcmFuZ2Uocm93cyk6CiAgICAgICAgICAgICAgICBjb2xzID0gbWluKGxlbihnYVtyXSksIGxlbihnYltyXSkpCiAgICAgICAgICAgICAgICBmb3IgYyBpbiByYW5nZShjb2xzKToKICAgICAgICAgICAgICAgICAgICBpZiBnYVtyXVtjXSAhPSBnYltyXVtjXSBhbmQgZ2Jbcl1bY10gIT0gZ2Fbcl1bY106CiAgICAgICAgICAgICAgICAgICAgICAgIGNlbGxzLmFwcGVuZCgociwgYykpCiAgICAgICAgICAgIGlmIGNlbGxzOgogICAgICAgICAgICAgICAgY29sb3JzID0ge30KICAgICAgICAgICAgICAgIGZvciBjZWxsIGluIGNlbGxzOgogICAgICAgICAgICAgICAgICAgIHYgPSBnYltjZWxsWzBdXVtjZWxsWzFdXQogICAgICAgICAgICAgICAgICAgIGlmIHYgaW4gY29sb3JzOgogICAgICAgICAgICAgICAgICAgICAgICBjb2xvcnNbdl0uYXBwZW5kKGNlbGwpCiAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgY29sb3JzW3ZdID0gW2NlbGxdCiAgICAgICAgICAgICAgICBiZXN0ID0gTm9uZQogICAgICAgICAgICAgICAgYmVzdF9uID0gMAogICAgICAgICAgICAgICAgZm9yIHYgaW4gY29sb3JzOgogICAgICAgICAgICAgICAgICAgIGlmIGxlbihjb2xvcnNbdl0pID4gYmVzdF9uOgogICAgICAgICAgICAgICAgICAgICAgICBiZXN0X24gPSBsZW4oY29sb3JzW3ZdKQogICAgICAgICAgICAgICAgICAgICAgICBiZXN0ID0gdgogICAgICAgICAgICAgICAgaWYgYmVzdCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBwdHMgPSBjb2xvcnNbYmVzdF0KICAgICAgICAgICAgICAgICAgICByID0gaW50KHJvdW5kKHN1bShwWzBdIGZvciBwIGluIHB0cykgLyBsZW4ocHRzKSkpCiAgICAgICAgICAgICAgICAgICAgYyA9IGludChyb3VuZChzdW0ocFsxXSBmb3IgcCBpbiBwdHMpIC8gbGVuKHB0cykpKQogICAgICAgICAgICAgICAgICAgIHJldHVybiBbciwgY10KICAgICAgICBpZHggPSBpZHggLSAxCiAgICByZXR1cm4gTm9uZQo=").decode("utf-8")
    _nav_note = _b64.b64decode("TkFWSUdBVElPTiBIRUxQRVJTIGFsc28gcHJlbG9hZGVkOiBtb3Rpb25fbW9kZWwoKSByZXR1cm5zIHRoZSBNRUFTVVJFRCB7YWN0aW9uOiBbZHJvdywgZGNvbF19IGxlYXJuZWQgZnJvbSB5b3VyIG93biB0cmFuc2l0aW9uczsgcGxhbl9tb3Zlcyhkcm93LCBkY29sKSByZXR1cm5zIGFuIG9yZGVyZWQgYWN0aW9uIGxpc3QgYWNoaWV2aW5nIHRoYXQgZGlzcGxhY2VtZW50OyBwbGF5ZXJfcG9zKCkgcmV0dXJucyB0aGUgW3JvdywgY29sXSBvZiB0aGUgb2JqZWN0IHRoYXQgbW92ZXMuIFByZWZlciBhY3Rpb24ocGxhbl9tb3ZlcyhkciwgZGMpKSB0byBleGVjdXRlIGEgd2hvbGUgcm91dGUgaW4gT05FIHR1cm4gaW5zdGVhZCBvZiBvbmUgYWN0aW9uIHBlciB0dXJuLg==").decode("utf-8")
    compile(_sh.SANDBOX_HELPERS_PRELUDE + "\n" + _nav_src, "<check>", "exec")
    _sh.SANDBOX_HELPERS_PRELUDE = _sh.SANDBOX_HELPERS_PRELUDE + "\n" + _nav_src
    _sh.HELPERS_PROMPT_NOTE = _sh.HELPERS_PROMPT_NOTE + "\n" + _nav_note
    print("NAV_HELPERS injected:", len(_nav_src), "chars")
except Exception as exc:
    print(f"[nav_helpers] injection failed, running stock: {type(exc).__name__}: {exc}")

import arc_agi, taaf.game_api
def games_offline(d):
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]
def games_comp():
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.COMPETITION,
                                    arc_base_url=os.environ["ARC_BASE_URL"], environments_dir="")
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.COMPETITION,
                        arc_base_url=spec.arc_base_url, environments_dir="")
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]

soft_end = None
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY","test-key-123")
    os.environ.setdefault("ARC_BASE_URL","http://gateway:8001/")
    dl = time.monotonic()+600
    while time.monotonic()<dl:
        try:
            with urlopen(os.environ["ARC_BASE_URL"]+"api/games", timeout=10) as r:
                if r.status<500: break
        except Exception: pass
        time.sleep(5)
    bm.games = games_comp()
else:
    bm.games = games_offline(str(COMP_ROOT/"environment_files"))
    soft_end = datetime.fromtimestamp(NOTEBOOK_START)+timedelta(minutes=OFFLINE_SOFT_MIN)

import pandas as pd
pd.DataFrame([["1_0","1",True,1]], columns=["row_id","game_id","end_of_game","score"]).to_parquet(WORKING/"submission.parquet", index=False)

try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
finally:
    for c in json.loads((BUNDLE/"teardown_commands.json").read_text()):
        subprocess.run(c, shell=True, check=False, cwd=WORKING, env=cmd_env())
print("run terminado")
